# 01 - 模型保存与加载

## 学习目标
- 掌握 sklearn 模型保存
- 掌握 TensorFlow 模型保存
- 了解模型版本管理

In [ ]:
import joblib
import pickle
import numpy as np
from pathlib import Path

from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## 1. sklearn 模型保存

In [ ]:
# 训练模型
data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# 创建 Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
print(f'训练准确率: {pipeline.score(X_test, y_test):.4f}')

In [ ]:
# 保存模型
save_dir = Path('../../models_saved/sklearn')
save_dir.mkdir(parents=True, exist_ok=True)

# 方法1: joblib (推荐)
joblib.dump(pipeline, save_dir / 'iris_classifier.pkl')
print(f'✓ 模型已保存到 {save_dir / "iris_classifier.pkl"}')

# 方法2: pickle
with open(save_dir / 'iris_classifier_pickle.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print(f'✓ 模型已保存到 {save_dir / "iris_classifier_pickle.pkl"}')

In [ ]:
# 加载模型
loaded_model = joblib.load(save_dir / 'iris_classifier.pkl')

# 测试
new_data = np.array([[5.1, 3.5, 1.4, 0.2]])  # 新样本
prediction = loaded_model.predict(new_data)
print(f'预测类别: {prediction[0]} ({data.target_names[prediction[0]]})')
print(f'预测概率: {loaded_model.predict_proba(new_data)[0]}')

## 2. TensorFlow 模型保存

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 构建并训练模型
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=3, verbose=0)

In [ ]:
# 保存模型
tf_save_dir = Path('../../models_saved/tensorflow')
tf_save_dir.mkdir(parents=True, exist_ok=True)

# 方法1: SavedModel 格式 (推荐)
model.save(tf_save_dir / 'mnist_model')
print(f'✓ SavedModel 格式已保存')

# 方法2: HDF5 格式
model.save(tf_save_dir / 'mnist_model.h5')
print(f'✓ HDF5 格式已保存')

# 方法3: 只保存权重
model.save_weights(tf_save_dir / 'mnist_weights')
print(f'✓ 权重已保存')

In [ ]:
# 加载模型
loaded_tf_model = keras.models.load_model(tf_save_dir / 'mnist_model')

# 测试
test_loss, test_acc = loaded_tf_model.evaluate(x_test, y_test, verbose=0)
print(f'加载模型测试准确率: {test_acc:.4f}')

## 3. 模型版本管理

In [ ]:
import json
from datetime import datetime

# 模型元数据
model_metadata = {
    'name': 'iris_classifier',
    'version': '1.0.0',
    'created_at': datetime.now().isoformat(),
    'metrics': {
        'accuracy': float(pipeline.score(X_test, y_test))
    },
    'params': pipeline.get_params(),
    'features': list(data.feature_names)
}

# 保存元数据
with open(save_dir / 'model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2, default=str)

print('模型元数据:')
print(json.dumps(model_metadata, indent=2, default=str))

## 小结
- joblib 是 sklearn 模型保存的推荐方式
- TensorFlow 推荐使用 SavedModel 格式
- 保存模型元数据便于版本管理